# SM $hhhh\to 8b$ vs. $gg\to 8b$: ROOT → XGBoost → pyhf

This notebook follows two Monte Carlo samples through a complete miniature analysis. It assumes that you have not performed this analysis before, so every transition—ROOT records, weights, folds, classifier scores, histograms, likelihoods, fits, and limits—is explained.

The whole chain is

$$
\text{ROOT events}\to\text{validated features and weights}\to
\text{cross-fitted XGBoost scores}\to\text{frozen score histograms}\to
\text{HistFactory workspace}\to\text{pyhf inference}.
$$

There are two distinct statistical tasks:

1. **XGBoost classification:** learn a ranking that separates signal-like from background-like events.
2. **pyhf inference:** use Poisson counts in fixed score bins to estimate or constrain a signal cross section while profiling systematic and finite-MC nuisance parameters.

> **XGBoost is trained first. pyhf does not fit, smooth, calibrate, or retrain the classifier.** “Fitting the XGBoost score” means fitting the bin counts in frozen templates of that score.

The headline result is a **background-only median expected** limit. A separate injected-Asimov construction is used only to make fit evolution visible and is always labelled **NOT DATA**. For a slower standalone explanation, read `BEGINNER_GUIDE.md` alongside this notebook.


## 0. Setup and reproducibility

Run this notebook from either the repository root or its own directory. ROOT/PyROOT must already be active; `requirements.txt` supplies the tutorial-specific Python layer. The helper validates all four input hashes before reading an event and records versions, seeds, resolved configuration, and hashes in `tutorial_outputs/`.

The thread-count environment variables below remove a common source of run-to-run variation in tree training and numerical linear algebra. The source notebook stored in Git contains no execution output; the separately generated `.executed.ipynb` is ignored.


In [ ]:
from pathlib import Path
import json
import math
import os
import sys

import numpy as np
import pandas as pd
from IPython.display import Image, display

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

def find_tutorial_dir():
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        if base.name == "sm_gg8b_xgboost_pyhf" and (base / "config.json").is_file():
            return base
        candidate = base / "tutorials" / "sm_gg8b_xgboost_pyhf"
        if (candidate / "config.json").is_file():
            return candidate
    raise FileNotFoundError("Could not locate tutorials/sm_gg8b_xgboost_pyhf")

TUTORIAL_DIR = find_tutorial_dir()
REPO_ROOT = TUTORIAL_DIR.parents[1]
sys.path.insert(0, str(TUTORIAL_DIR))

from tutorial_helpers import load_config, run_tutorial

CONFIG_PATH = TUTORIAL_DIR / "config.json"
OUTPUT_DIR = TUTORIAL_DIR / "tutorial_outputs"
config = load_config(CONFIG_PATH)
print(f"Repository: {REPO_ROOT}")
print(f"Configuration: {CONFIG_PATH}")
print(f"Outputs: {OUTPUT_DIR}")

## 1. Physics normalization and the ROOT contract

The `Data3` tree uses schema `extended-91-v2`. This lesson selects the named `corrected28` projection, giving 28 kinematic classifier inputs. A stored event also has a signed Monte Carlo weight and stable identifiers, but the weight is deliberately kept outside the feature matrix.

For an accepted event with stored weight $w_i$,

$$
w_i^{\rm phys}=\mathcal L\,\sigma\,F\,\frac{w_i}{W_{\rm input}},
$$

where $W_{\rm input}$ is read from the matching analysis-summary JSON. The fixed final-state factors are

$$
F_s=K_s\,{\rm BR}(h\to b\bar b)^4\epsilon_b^8,\qquad
F_b=K_b\,\epsilon_b^8.
$$

Here $K_s=K_b=2$ is the configured phenomenological $K$-factor convention used by this analysis; it is not a combinatorial factor or a fitted uncertainty. The four powers of the branching ratio describe four Higgs decays, and the eight powers of $\epsilon_b$ represent the idealized requirement to tag eight $b$ jets.

We retain four different notions of weight:

- **raw:** the signed value stored in ROOT;
- **physical:** the expected contribution at the declared luminosity and cross section;
- **unit-cross-section signal:** the signal yield with $\sigma_{hhhh}=1$ fb;
- **training:** a non-negative, class-balanced value proportional to $|w_i^{\rm phys}|$.

Only the training weight enters the XGBoost loss. Signed physical or unit-cross-section weights enter yields. In a bin,

$$
N=\sum_i w_i,\qquad \delta N_{\rm MC}=\sqrt{\sum_i w_i^2}.
$$

A negative event can cancel yield, but it still contributes positively to MC variance. The signal unit-cross-section template later makes the pyhf POI numerically equal to the production cross section `sigma_hhhh_fb`, before applying the fixed $K_s$.


In [ ]:
signal_factor = (
    config.signal.k_factor
    * config.branching_ratio_hbb**4
    * config.btag_efficiency**8
)
background_factor = (
    config.background.k_factor
    * config.btag_efficiency**8
)
pd.DataFrame(
    {
        "quantity": [
            "sqrt(s) [TeV]",
            "L [fb^-1]",
            "sigma_SM [fb]",
            "K_signal",
            "K_background",
            "F_signal",
            "F_background",
        ],
        "value": [
            config.sqrt_s_tev,
            config.luminosity_fb,
            config.sm_cross_section_fb,
            config.signal.k_factor,
            config.background.k_factor,
            signal_factor,
            background_factor,
        ],
    }
)


## 2. Execute the deterministic analysis

The next call performs the complete auditable sequence:

1. verify hashes, sidecars, schema, projection, event identifiers, and absence of weight leakage;
2. load both ROOT samples and build raw, physical, unit-cross-section, and training weights;
3. run five XGBoost rotations, with three training folds, one bin-selection fold, and one inference fold;
4. select score bins using validation events only, freeze the edges, and fill held-out test templates;
5. construct the five-channel and inclusive one-bin HistFactory workspaces;
6. fit the background-only Asimov model, calculate expected limits, and fit a separately labelled injected Asimov example;
7. save event scores, fold models, JSON workspaces/results, and PDF/PNG figures.

No Optuna search occurs here. Hyperparameters and seeds are fixed before looking at the held-out events. The call can take several minutes because it trains five classifiers and performs many profiled likelihood fits.


In [ ]:
result = run_tutorial(CONFIG_PATH, output_dir=OUTPUT_DIR, make_plots=True)
signal, background = result["samples"]
print("Complete. Headline median expected limit:", result["fit_results"]["headline_expected_limit"]["expected_median_fb"], "fb")

### Inspect the loaded events and check normalization

The table below performs an **internal arithmetic and metadata closure**, not an independent validation of the cross section. It recomputes $F$ directly from the primitive configuration values, rather than reusing the helper's stored `rate_factor`, and checks

$$
\sum_iw_i^{\rm phys}
=\mathcal L\,\sigma\,F\,\frac{\sum_iw_i}{W_{\rm input}}.
$$

Agreement proves that the declared inputs were applied consistently. It does not prove that the cross section, $K$-factor, branching ratio, or tagging model is physically complete; those are assumptions supplied to the calculation.

The feature assertion is a separate leakage check: no branch whose name contains “weight” may appear among the 28 classifier features.


In [ ]:
rows = []
for sample in (signal, background):
    if sample.spec.is_signal:
        factor_from_primitives = (
            sample.spec.k_factor
            * config.branching_ratio_hbb**4
            * config.btag_efficiency**8
        )
    else:
        factor_from_primitives = (
            sample.spec.k_factor
            * config.btag_efficiency**8
        )
    closure = (
        config.luminosity_fb
        * sample.spec.cross_section_fb
        * factor_from_primitives
        * sample.raw_weights.sum()
        / sample.total_weight_in
    )
    assert np.isclose(factor_from_primitives, sample.rate_factor)
    assert np.isclose(closure, sample.physical_weights.sum())
    rows.append(
        {
            "sample": sample.spec.name,
            "events": sample.entries,
            "features": sample.features.shape[1],
            "K factor": sample.spec.k_factor,
            "F from primitives": factor_from_primitives,
            "total_weight_in": sample.total_weight_in,
            "sum physical weights": sample.physical_weights.sum(),
            "arithmetic closure": closure,
            "SHA-256 (ROOT)": sample.hashes["root"][:12] + "…",
        }
    )

feature_names = result["crossfit"]["feature_names"]
assert len(feature_names) == 28
assert all("weight" not in name.lower() for name in feature_names)
display(pd.DataFrame(rows))
display(pd.DataFrame({"classifier feature": feature_names}))


## 3. XGBoost cross-fitting: why the folds matter

A classifier evaluated on its own training events can exploit statistical accidents. Its apparent separation—and hence a limit based on that separation—can be too optimistic. Cross-fitting ensures that every event used in the likelihood is scored by a model that did not train on it.

In rotation $r$, fold $r$ is the test/inference fold, fold $(r+1)\bmod5$ selects score bins, and the other three folds train XGBoost:

| rotation | inference | bin selection | training |
| ---: | ---: | ---: | --- |
| 0 | 0 | 1 | 2, 3, 4 |
| 1 | 1 | 2 | 0, 3, 4 |
| 2 | 2 | 3 | 0, 1, 4 |
| 3 | 3 | 4 | 0, 1, 2 |
| 4 | 4 | 0 | 1, 2, 3 |

Training minimizes weighted binary log loss using non-negative $|w_i^{\rm phys}|$, rescaled so signal and background have equal total training weight. That class balancing keeps the much larger background normalization from dominating the classifier loss. Signed weights are retained separately for physical histograms.

Each event receives exactly one **out-of-fold** test score. The five disjoint test folds cover all events, so no event or signed yield is lost.

There is a subtle limitation worth stating precisely. A validation event never chooses the edges for its *own* channel. However, fold $f+1$ selects channel $f$ and later enters the combined likelihood as test channel $f+1$. Since the final likelihood contains all folds, bin choice and the complete combined dataset are not strictly independent. This can cause mild selection-induced optimism. A separate selection sample or nested cross-validation would be required for exact independence.


In [ ]:
fold_rows = []
test_multiplicity = np.zeros(len(result["crossfit"]["oof_scores"]), dtype=int)
for rotation in result["crossfit"]["rotations"]:
    test_multiplicity += rotation["test_mask"]
    fold_rows.append(
        {
            "rotation": rotation["rotation"],
            "train folds": rotation["train_folds"],
            "validation fold": rotation["validation_fold"],
            "test fold": rotation["test_fold"],
            "N train/val/test": f"{rotation['n_train']}/{rotation['n_validation']}/{rotation['n_test']}",
            "class weight S": rotation["classifier_signal_weight"],
            "class weight B": rotation["classifier_background_weight"],
        }
    )
assert np.all(test_multiplicity == 1)
assert np.all(np.isfinite(result["crossfit"]["oof_scores"]))
pd.DataFrame(fold_rows)

In [ ]:
display(Image(filename=result["plots"]["weighted_roc"]["png"], width=620))
display(Image(filename=result["plots"]["feature_importance"]["png"], width=720))

## 4. Freeze the score bins before inference

The XGBoost output is a ranking score, not necessarily a calibrated probability. We retain shape information by dividing it into a few statistically stable regions.

Each validation fold proposes two-to-five bins from background quantiles

$$
[0,\ 0.50,\ 0.75,\ 0.90,\ 0.97,\ 1].
$$

Every validation bin must have positive signed background yield, at least 25 raw background entries, and

$$
N_{\rm eff}=\frac{(\sum_iw_i)^2}{\sum_iw_i^2}\ge10.
$$

$N_{\rm eff}$ measures weighted statistical precision; cancellations can make it much smaller than the raw count. Candidates within 1% of the best expected sensitivity prefer fewer bins, guarding against an unstable gain from finely slicing the high-score tail.

One validation fold is multiplied by five to represent a full-sample yield while choosing bins. This sends $\sum w\to5\sum w$ and $\sum w^2\to25\sum w^2$, so the *relative* MC uncertainty stays at its one-fold value. A genuine five-fold independent sample would have a relative uncertainty smaller by $\sqrt5$. The surrogate is therefore conservative and can favor coarser binning.

Only after selection are edges frozen and applied to the test fold. A nonpositive signed test yield is never clipped. The code follows only the nested coarsening hierarchy declared by validation data, or stops if no valid template remains.


In [ ]:
bin_rows = []
for record, channel in zip(result["binning"]["records"], result["channels"]):
    selected = record["validation"]["selected"]
    bin_rows.append(
        {
            "fold": record["rotation"],
            "validation bins": selected["n_bins"],
            "test bins": len(channel["signal"]),
            "fallback level": record["chosen_fallback_level"],
            "min raw B (validation)": min(selected["background_raw_entries"]),
            "min Neff B (validation)": min(selected["background_effective_entries"]),
            "sum s / fb": np.sum(channel["signal"]),
            "sum b": np.sum(channel["background"]),
            "edges": np.round(record["edges"], 5).tolist(),
        }
    )
pd.DataFrame(bin_rows)

In [ ]:
display(Image(filename=result["plots"]["score_normalized"]["png"], width=700))
display(Image(filename=result["plots"]["score_expected_yields"]["png"], width=700))

## 5. Turn frozen histograms into a HistFactory workspace

Each held-out fold becomes one pyhf **channel**. In score bin $i$, its signal and background samples contain

$$
s_i=\sum_{\rm sig}w_i^{(1\,{\rm fb})},\quad
b_i=\sum_{\rm bkg}w_i^{\rm phys},\quad
\delta s_i=\sqrt{\sum_{\rm sig}w_i^2},\quad
\delta b_i=\sqrt{\sum_{\rm bkg}w_i^2}.
$$

A HistFactory workspace is a JSON description of:

- **channels:** disjoint collections of bins, here the five held-out folds;
- **samples:** signal and background templates inside each channel;
- **observations:** the bin values to be fitted;
- **modifiers:** parameters that change sample expectations;
- **a measurement:** the POI declaration, initial values, and bounds.

This workspace uses:

- a signal `normfactor` named `sigma_hhhh_fb`, shared across all channels and unconstrained by an auxiliary measurement;
- independent, channel-local signal and background `staterror` terms built from $\sqrt{\sum w^2}$;
- one shared pedagogical background `normsys`, `gg8b_norm`, with `lo=0.90` and `hi=1.10`.

The recorded 2.54% generation integration error is provenance, not a QCD theory nuisance. The 10% `gg8b_norm` term is also illustrative, not a measured uncertainty.

For the headline expected result, the observations are the exact nominal background expectations. This is a **background-only Asimov dataset**: a deterministic, no-fluctuation dataset evaluated at $\sigma_{hhhh}=0$ and nominal nuisances. Asimov bin values may be fractional. “Observed-on-Asimov” is not observed collision data; on this construction it normally equals the median expected result.


In [ ]:
workspace_spec = result["workspace"]
print(json.dumps({"measurement": workspace_spec["measurements"][0], "first channel": workspace_spec["channels"][0]}, indent=2)[:7000])

In [ ]:
import pyhf

pyhf.set_backend("numpy", pyhf.optimize.scipy_optimizer())
workspace = pyhf.Workspace(workspace_spec)
model = workspace.model(measurement_name="sm_hhhh_measurement")
data = workspace.data(model)

print("POI:", model.config.poi_name)
print("main/auxiliary data:", model.config.nmaindata, model.config.nauxdata)
print("number of fit parameters:", model.config.npars)
print("first parameter names:", model.config.par_names[:12])
assert len(data) == model.config.nmaindata + model.config.nauxdata

Schematically, pyhf evaluates

$$
\mathcal L(\sigma,\boldsymbol\theta)
=\prod_i {\rm Pois}\!\left[n_i\mid
\sigma s_i(\boldsymbol\theta)+b_i(\boldsymbol\theta)\right]
\prod_k\pi_k(\theta_k).
$$

The first product compares the observed/Asimov score-bin counts with their expected values. The second contains auxiliary constraints for finite-MC and normalization nuisance parameters. This is why `workspace.data(model)` has both `nmaindata` score-bin entries and `nauxdata` constraint entries.

The **global maximum-likelihood fit** varies $\sigma$ and all nuisance parameters. A **fixed-POI fit** holds $\sigma$ at one proposed value while re-optimizing, or *profiling*, all nuisances. The profile likelihood therefore asks how well a proposed signal can fit after every allowed nuisance adjustment.

pyhf sees only the frozen arrays $s_i$, $b_i$, their uncertainties, and the workspace modifiers. It never sees the original 28 event variables and cannot change the XGBoost model.


## 6. Use the pyhf inference APIs directly

The next cell exposes the main calls rather than hiding them behind the tutorial driver:

- `pyhf.infer.mle.fit` finds the global MLE $(\hat\sigma,\hat{\boldsymbol\theta})$;
- `fixed_poi_fit` fixes $\sigma$ and profiles all nuisance parameters;
- `hypotest(..., test_stat="qtilde")` evaluates asymptotic $CL_s$ for a proposed cross section;
- `upper_limits.upper_limit` finds a crossing on an explicit scan;
- `upper_limits.toms748_scan` root-finds a more precise crossing.

The bounded $\widetilde q_\mu$ statistic is appropriate because a physical signal strength cannot be negative. A 95% confidence level (95% CL) upper limit is the cross section where

$$
CL_s(\sigma_{95})=0.05.
$$

The five expected limits correspond to background-only outcomes at $-2\sigma,-1\sigma,$ median, $+1\sigma,+2\sigma$. They are the expected spread across hypothetical repeated background-only experiments, not an uncertainty bar on one measured limit.

**pyhf 0.7.6 detail:** with `scan=None`, `upper_limit(..., level=...)` does not forward a non-default level to its internal root finder. We therefore use an explicit scan when teaching `upper_limit` and call `toms748_scan` directly for the precise reported answer, with `level = 1 - confidence_level`. A regression test compares 90% and 95% limits.


In [ ]:
confidence_level = 0.95
alpha = 1.0 - confidence_level

# 1. Global fit: POI and every nuisance are free.
initial_parameters = model.config.suggested_init()
initial_parameters[model.config.poi_index] = 0.0
bestfit, twice_nll_free = pyhf.infer.mle.fit(
    data,
    model,
    init_pars=initial_parameters,
    return_fitted_val=True,
)
bestfit = np.asarray(bestfit, dtype=float)
sigma_hat = bestfit[model.config.poi_index]

# 2. Conditional fit: fix the POI at the median expected limit and profile nuisances.
sigma95 = result["fit_results"]["headline_expected_limit"]["expected_median_fb"]
fixed_at_limit, twice_nll_fixed = pyhf.infer.mle.fixed_poi_fit(
    sigma95,
    data,
    model,
    return_fitted_val=True,
)
fixed_at_limit = np.asarray(fixed_at_limit, dtype=float)
delta_nll_at_limit = 0.5 * (
    float(np.asarray(twice_nll_fixed)) - float(np.asarray(twice_nll_free))
)

# 3. Test that same proposed cross section with qtilde.
cls_on_asimov, cls_expected = pyhf.infer.hypotest(
    sigma95,
    data,
    model,
    test_stat="qtilde",
    return_expected_set=True,
)

# 4. Exact root-finding used for the reported result.
poi_bounds = model.config.suggested_bounds()[model.config.poi_index]
limit_on_asimov, expected_bands = (
    pyhf.infer.intervals.upper_limits.toms748_scan(
        data,
        model,
        float(poi_bounds[0]),
        float(poi_bounds[1]),
        level=alpha,
        test_stat="qtilde",
    )
)

# 5. The documented upper_limit convenience API, with an explicit finite grid.
#    Linear interpolation makes this illustrative value less precise than toms748.
scan = np.linspace(0.0, 1.25 * float(np.asarray(expected_bands)[-1]), 17)
grid_limit, grid_expected_bands = (
    pyhf.infer.intervals.upper_limits.upper_limit(
        data,
        model,
        scan=scan,
        level=alpha,
        test_stat="qtilde",
    )
)

interesting_parameters = [
    model.config.poi_name,
    "gg8b_norm",
]
fit_rows = []
for name in interesting_parameters:
    parameter_slice = model.config.par_slice(name)
    index = parameter_slice.start
    fit_rows.append(
        {
            "parameter": name,
            "global MLE": bestfit[index],
            "fixed-POI conditional fit": fixed_at_limit[index],
        }
    )

print(f"background-only global MLE: sigma_hat = {sigma_hat:.8g} fb")
print(f"fixed-POI value: sigma = {sigma95:.8g} fb")
print(f"Delta(-log L) at that fixed value: {delta_nll_at_limit:.6g}")
print(f"qtilde CLs there (Asimov): {float(np.asarray(cls_on_asimov)):.6g}")
print("expected CLs set [-2,-1,median,+1,+2 sigma]:", np.asarray(cls_expected))
print(f"observed-on-background-Asimov 95% limit: {float(np.asarray(limit_on_asimov)):.8g} fb")
print("precise expected limit bands [fb]:", np.asarray(expected_bands))
print(f"explicit-grid upper_limit median (approximate): {float(np.asarray(grid_expected_bands)[2]):.8g} fb")
display(pd.DataFrame(fit_rows))

assert np.isclose(float(np.asarray(expected_bands)[2]), sigma95, rtol=5e-4)
assert np.isclose(float(np.asarray(cls_on_asimov)), alpha, rtol=5e-3)


In [ ]:
limit = result["fit_results"]["headline_expected_limit"]
control = result["fit_results"]["one_bin_control"]
mcstat_only = result["fit_results"]["shape_mcstat_only"]
pd.DataFrame(
    [
        {"model": "inclusive one-bin control", "median sigma95 [fb]": control["expected_median_fb"], "median mu95": control["expected_median_fb"] / config.sm_cross_section_fb},
        {"model": "five-channel shape (MC stat only)", "median sigma95 [fb]": mcstat_only["expected_median_fb"], "median mu95": mcstat_only["expected_median_mu"]},
        {"model": "five-channel shape (+ pedagogical 10% norm)", "median sigma95 [fb]": limit["expected_median_fb"], "median mu95": limit["expected_median_mu"]},
    ]
)

In [ ]:
display(Image(filename=result["plots"]["cls_scan"]["png"], width=700))

## 7. Make fit evolution visible with an injected Asimov sample — NOT DATA

A background-only Asimov fit is intentionally uneventful: prefit and postfit agree and $\hat\sigma\simeq0$. To display a meaningful movement, the tutorial generates a *separate* conditional Asimov vector at

$$
\sigma_{\rm inj}=0.5\,\sigma_{95}^{\rm expected}
$$

and a $+0.5\sigma$ displacement of `gg8b_norm`. Its main-bin expectation includes both effects, and its auxiliary measurement is displaced consistently.

Because the data vector is generated deterministically from the same model being fitted, a converged optimizer should recover the injected POI and nuisance to numerical precision. This is an implementation-closure test, not evidence that the model describes nature.

In the unrolled figure:

- the blue curve is the nominal background-only prefit expectation;
- the orange curve and black points are the full injected generating expectation, including signal **and** the normalization shift;
- the green curve is the postfit expectation;
- the ratio compares deterministic Asimov bin values with the fitted expectation, so no pseudo-Poisson error bars are assigned to the Asimov points.

This sample is not observed data and does not enter the headline limit.


In [ ]:
injection = result["fit_results"]["injected_asimov"]
fit = injection["fit"]
truth = np.asarray(injection["truth_parameters"], dtype=float)
fitted = np.asarray(fit["bestfit_parameters"], dtype=float)
parameter_names = fit["parameter_names"]

poi_index = parameter_names.index("sigma_hhhh_fb")
norm_index = parameter_names.index("gg8b_norm")
recovery = pd.DataFrame(
    [
        {
            "parameter": "sigma_hhhh_fb",
            "injected": truth[poi_index],
            "fitted": fitted[poi_index],
            "difference": fitted[poi_index] - truth[poi_index],
        },
        {
            "parameter": "gg8b_norm [constraint sigma]",
            "injected": truth[norm_index],
            "fitted": fitted[norm_index],
            "difference": fitted[norm_index] - truth[norm_index],
        },
    ]
)

print(injection["label"])
print("fit backend:", fit["backend"])
print("is observed data?", injection["is_observed_data"])
display(recovery)
assert injection["is_observed_data"] is False
assert np.isclose(fitted[poi_index], truth[poi_index], rtol=2e-5)
assert np.isclose(fitted[norm_index], truth[norm_index], atol=1e-3)


In [ ]:
display(Image(filename=result["plots"]["unrolled_fit"]["png"], width=950))
display(Image(filename=result["plots"]["likelihood_scan"]["png"], width=680))
display(Image(filename=result["plots"]["nuisance_pulls"]["png"], width=760))
display(Image(filename=result["plots"]["reduced_correlation"]["png"], width=760))

## 8. Auditable outputs

The event table records source/event identifiers, folds, raw/signed/unit-cross-section weights, and exactly one out-of-fold score. The repository also saves:

- five independently trained fold models and their metadata;
- selected and fallback binning for every rotation;
- the primary shape, MC-stat-only comparison, and inclusive one-bin workspaces;
- global/injected fits, profile and $CL_s$ scans, expected limits, pulls, and correlations;
- resolved configuration, package versions, seeds, and input hashes;
- vector PDF and 300-dpi PNG versions of every requested figure.

These artifacts let another reader trace a plotted bin back to scored events and normalization inputs rather than treating the notebook output as an opaque final number.


In [ ]:
artifact_rows = []
for name, path in sorted(result["artifacts"].items()):
    item = Path(path)
    artifact_rows.append({"artifact": name, "path": str(item.relative_to(TUTORIAL_DIR)), "bytes": item.stat().st_size})
pd.DataFrame(artifact_rows)

## 9. Interpretation, limitations, and relation to production `fast-sm`

This lesson emulates the central production architecture:

- complete-event deterministic five-fold SM cross-fitting;
- fixed classifier models and exactly-once out-of-fold scores;
- signed physical yields and 1-fb signal templates;
- validation-defined score regions;
- pyhf score-shape likelihoods and expected limits.

It is deliberately smaller and is **not numerically identical** to `fast-sm`:

- this notebook uses `corrected28`; production `fast-sm` uses `full91` for `extended-91-v2`;
- this notebook includes only SM $hhhh\to8b$ and pure $gg\to8b$;
- production uses its configured full background composition and broader coupling-point machinery;
- the 10% `gg8b_norm` nuisance and injected Asimov sample are pedagogical;
- the validation-fold yield surrogate retains conservative one-fold relative MC statistics.

The two-sample result is not a publication-ready exclusion. It omits reducible/mistag backgrounds, genuine detector and theory shape variations, and observed collision data. Omitting positive background components will generally make an expected limit look too strong, although changed training and score shapes prevent a universal rescaling. The 2.54% generation integration error is recorded but is not interpreted as a QCD uncertainty.

The plotting machinery is designed for later reuse: neutral HEP typography, colorblind-safe colors, explicit energy/luminosity/Asimov labels, PDF and 300-dpi PNG output, MC-stat bands, an unrolled ratio panel, a profile-likelihood scan, expected $CL_s$ bands, nuisance pulls, and a POI-focused correlation matrix.

The correct headline sentence is:

> Under the explicitly declared two-sample HistFactory model, the median background-only Asimov 95% CL upper limit is the value stored in `fit_results.json`; it is an expected teaching result, not an observed exclusion.
